# Downloading the AGAP airborne magnetic survey to use in the notebooks

Accessed from https://data.bas.ac.uk/full-record.php?id=GB/NERC/BAS/PDC/01308

In [ ]:
import pandas as pd
import pooch

import airbornegeo

## load data

In [ ]:
path = pooch.retrieve(
    url="https://ramadda.data.bas.ac.uk/repository/entry/get/AGAP_Mag.XYZ?entryid=synth%3A91df43df-e3fd-4637-9091-e75f623e2b07%3AL0FHQVBfTWFnLlhZWg%3D%3D",
    fname="AGAP_Mag.XYZ",
    path=f"{pooch.os_cache('airbornegeo')}",
    known_hash="53e3072cb4f2abdd5f37288d05ab40d040506b6939fcdd6772b0e20dff6b7a01",
    progressbar=True,
)
data_df = pd.read_csv(path)
data_df

In [ ]:
skiprow = list(range(11))
skiprow.remove(5)
file = pd.read_csv(path, skiprows=skiprow, sep=" +", engine="python")
for i, col in enumerate(file.columns):
    print(i - 1, col)

In [ ]:
data_df = pd.read_csv(
    path,
    skiprows=list(range(11)),
    sep=r"\s+",
    engine="python",
    na_values=["*"],
    names=file.columns[1:],
)

# drop projected coordinates, we will reproject ourselves
data_df = data_df.drop(columns=["x", "y"])

In [ ]:
data_df["easting"], data_df["northing"] = airbornegeo.reproject(
    data_df,
    input_crs="EPSG:4326",
    output_crs="EPSG:3031",
    input_coord_names=("Lon", "Lat"),
)

In [ ]:
# drop rows with nans in certain columns
data_df = data_df.dropna(subset=["Line_no", "Flight_ID", "Lon", "Lat"], how="any")

# combine flight and line strings together
data_df["line_name"] = data_df.Flight_ID + "_" + data_df.Line_no

# convert line label to integer
data_df["line"] = airbornegeo.unique_line_id(data_df, line_col_name="line_name")

data_df

In [ ]:
airbornegeo.plotly_points(
    data_df[::10],  # plot every 10th point
    color_col="line",
    hover_cols=["line_name"],
    robust=False,
    size=3,
    edge_width=None,
)

In [ ]:
# drop a few lines to clean up the survey
data_df = data_df[
    ~data_df.line_name.isin(
        [
            "BYRD1_69.0",
            "BYRD2_69.0",
            "BYRD3_69.0",
            "TAM1_4.0",
            "CTAM1_5.0",
            "SP10030_65.0",
            "F10291_37.0",
            "V10250_41.0",
        ]
    )
]

In [ ]:
# convert supplied line names and flights into integers
data_df["line"] = airbornegeo.unique_line_id(data_df, line_col_name="line_name")

# drop unneeded columns
data_df = data_df.drop(columns=["Flight_ID", "Line_no"])

# drop rows with all NaNs
data_df = data_df.dropna(how="all")

data_df

In [ ]:
airbornegeo.plotly_points(
    data_df[::10],
    color_col="line",
    hover_cols=["line_name"],
    robust=False,
    size=3,
    edge_width=None,
)

In [ ]:
# calculate the unixtime from the date and time columns
data_df["Date"] = pd.to_datetime(data_df["Date"])
data_df["Time"] = pd.to_timedelta(data_df["Time"])
data_df["unixtime"] = data_df["Date"] + data_df["Time"]
data_df = data_df.dropna(subset=["unixtime"])
data_df["unixtime"] = data_df["unixtime"].apply(lambda x: x.timestamp())
data_df = data_df.sort_values(["line", "unixtime"]).reset_index(drop=True)
data_df.head()

In [ ]:
airbornegeo.plotly_points(
    data_df[::10],
    color_col="unixtime",
    hover_cols=["line"],
    robust=False,
    size=3,
    edge_width=None,
)

In [ ]:
data_df.to_csv("../data/AGAP_magnetic_survey.csv", index=False)